In [1]:
import re

def split_into_sentences(text: str) -> list[str]:
    sentence_endings = re.compile(r'(?<=[.!?])\s+(?=[A-Z])')
    sentences = sentence_endings.split(text.strip())
    return [s.strip() for s in sentences if s.strip()]

sample_text = """Redis caching layer improves latency by 40%. However, this approach has tradeoffs. The team decided to migrate to gpt-oss-120b."""

sentences = split_into_sentences(sample_text)
for i, s in enumerate(sentences):
    print(f"{i}: {s}")

0: Redis caching layer improves latency by 40%.
1: However, this approach has tradeoffs.
2: The team decided to migrate to gpt-oss-120b.


In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embeddings(sentences: list[str]) -> np.ndarray:
    embeddings = model.encode(sentences)
    return embeddings

sentences = [
    "Redis caching layer improves latency by 40%.",
    "However, this approach has tradeoffs.",
    "The team decided to migrate to gpt-oss-120b."
]

embeddings = get_embeddings(sentences)
print(f"Shape: {embeddings.shape}")  

/Users/niteshkumar/RAG/DocumentLoaders/dloader/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6458.47it/s]


Shape: (3, 384)


In [4]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def compute_adjacent_similarities(embeddings: np.ndarray) -> list[float]:
    similarities = []
    for i in range(len(embeddings) - 1):
        sim = cosine_similarity(
            embeddings[i].reshape(1, -1),
            embeddings[i + 1].reshape(1, -1)
        )[0][0]
        similarities.append(sim)
    return similarities

similarities = compute_adjacent_similarities(embeddings)
for i, sim in enumerate(similarities):
    print(f"Sentence {i} <-> Sentence {i+1}: similarity = {sim:.4f}")

Sentence 0 <-> Sentence 1: similarity = 0.2035
Sentence 1 <-> Sentence 2: similarity = 0.0238


In [5]:
similarities

[np.float32(0.2035028), np.float32(0.023753557)]

In [6]:

from src.chunker import TextSemanticChunker

In [7]:
splitter = TextSemanticChunker()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5530.76it/s]


In [10]:
documents = splitter.chunk(sentences)

In [11]:
documents

[Document(metadata={}, page_content='Redis caching layer improves latency by 40%.'),
 Document(metadata={}, page_content='However, this approach has tradeoffs.'),
 Document(metadata={}, page_content='The team decided to migrate to gpt-oss-120b.')]

In [2]:
from rank_bm25 import BM25Okapi

corpus = [
    "Redis caching layer improves latency by significant margins.",
    "Error E4021 indicates a Redis connection timeout issue.",
    "The team decided to migrate to gpt-oss-120b model.",
    "PostgreSQL handles relational data with strong consistency."
]

tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

query = "Redis connection timeout error E4021"
tokenized_query = query.lower().split()

scores = bm25.get_scores(tokenized_query)
for i, score in enumerate(scores):
    print(f"Doc {i}: {score:.4f} — {corpus[i]}")

Doc 0: 0.0000 — Redis caching layer improves latency by significant margins.
Doc 1: 3.3407 — Error E4021 indicates a Redis connection timeout issue.
Doc 2: 0.0000 — The team decided to migrate to gpt-oss-120b model.
Doc 3: 0.0000 — PostgreSQL handles relational data with strong consistency.


In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

corpus = [
    "Redis caching layer improves latency by significant margins.",
    "Error E4021 indicates a Redis connection timeout issue.",
    "The team decided to migrate to gpt-oss-120b model.",
    "PostgreSQL handles relational data with strong consistency."
]

corpus_embeddings = model.encode(corpus)
query = "Redis connection timeout error E4021"
query_embedding = model.encode([query])

vector_scores = cosine_similarity(query_embedding, corpus_embeddings)[0]

for i, score in enumerate(vector_scores):
    print(f"Doc {i}: {score:.4f} — {corpus[i]}")

/Users/niteshkumar/RAG/DocumentLoaders/dloader/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5502.72it/s]


Doc 0: 0.4046 — Redis caching layer improves latency by significant margins.
Doc 1: 0.9501 — Error E4021 indicates a Redis connection timeout issue.
Doc 2: 0.0904 — The team decided to migrate to gpt-oss-120b model.
Doc 3: 0.0205 — PostgreSQL handles relational data with strong consistency.


In [4]:
def reciprocal_rank_fusion(bm25_scores, vector_scores, k=60):
    num_docs = len(bm25_scores)
    bm25_ranks = np.argsort(-np.array(bm25_scores)) 
    vector_ranks = np.argsort(-np.array(vector_scores))
    
    
    bm25_rank_map = {doc_id: rank for rank, doc_id in enumerate(bm25_ranks)}
    vector_rank_map = {doc_id: rank for rank, doc_id in enumerate(vector_ranks)}
    
    rrf_scores = {}
    for doc_id in range(num_docs):
        bm25_contribution = 1 / (k + bm25_rank_map[doc_id] + 1) 
        vector_contribution = 1 / (k + vector_rank_map[doc_id] + 1)
        rrf_scores[doc_id] = bm25_contribution + vector_contribution
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_docs


final_ranking = reciprocal_rank_fusion(scores, vector_scores)
for doc_id, rrf_score in final_ranking:
    print(f"Doc {doc_id}: RRF={rrf_score:.5f} — {corpus[doc_id]}")

Doc 1: RRF=0.03279 — Error E4021 indicates a Redis connection timeout issue.
Doc 0: RRF=0.03226 — Redis caching layer improves latency by significant margins.
Doc 2: RRF=0.03175 — The team decided to migrate to gpt-oss-120b model.
Doc 3: RRF=0.03125 — PostgreSQL handles relational data with strong consistency.
